In [2]:
!pip install faker

     ---------------------------------------- 1.9/1.9 MB 629.7 kB/s eta 0:00:00
     -------------------------------------- 347.8/347.8 kB 1.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import random
from faker import Faker

In [5]:
# Initialize
fake = Faker()
random.seed(42)
np.random.seed(42)

In [6]:
# Sample data lists
loan_types = ['Retail', 'MSME', 'Corporate', 'Agriculture', 'Housing', 'Education', 'Personal']
industries = ['Manufacturing', 'Services', 'Trading', 'Agriculture', 'Infrastructure', 'Healthcare', 'Retail Trade']
regions = ['North', 'South', 'East', 'West', 'Central', 'Northeast']
loan_purposes = ['Working Capital', 'Term Loan', 'Project Finance', 'Personal Use', 'Housing Loan', 'Education Loan']
branches = ['Mumbai', 'Delhi', 'Bengaluru', 'Chennai', 'Hyderabad', 'Kolkata', 'Lucknow', 'Indore', 'Ahmedabad', 'Chandigarh']

npa_mapping = {
    'Standard': 0.004,
    'Substandard': 0.15,
    'Doubtful-1': 0.25,
    'Doubtful-2': 0.4,
    'Doubtful-3': 1.0,
    'Loss': 1.0
}

In [7]:
def get_sma_status(dpd):
    if dpd <= 30:
        return 'SMA-0'
    elif 31 <= dpd <= 60:
        return 'SMA-1'
    elif 61 <= dpd <= 90:
        return 'SMA-2'
    else:
        return 'NPA'

In [8]:
def get_asset_classification(dpd):
    if dpd <= 90:
        return 'Standard'
    elif 91 <= dpd <= 180:
        return 'Substandard'
    elif 181 <= dpd <= 365:
        return 'Doubtful-1'
    elif 366 <= dpd <= 720:
        return 'Doubtful-2'
    elif 721 <= dpd <= 1095:
        return 'Doubtful-3'
    else:
        return 'Loss'

In [14]:
def generate_npa_data(n=10000):
    data = []
    for i in range(n):
        loan_id = f"LN{i+1:06d}"
        loan_type = random.choice(loan_types)
        loan_amount = round(random.uniform(50000, 10_00_00_000), 2)
        overdue_percentage = random.uniform(0.2, 1.0)
        overdue_amount = round(loan_amount * overdue_percentage, 2)
        interest_rate = round(random.uniform(7, 18), 2)
        tenure_months = random.randint(12, 240)
        disb_date = fake.date_between(start_date='-5y', end_date='today')
        borrower_id = f"BR{i+1:06d}"
        credit_score = random.randint(300, 850)
        annual_income = random.randint(1_00_000, 5_00_00_000)
        industry = random.choice(industries)
        region = random.choice(regions)
        branch_name = random.choice(branches)
        loan_purpose = random.choice(loan_purposes)
        days_past_due = random.randint(0, 1100)
        sma_status = get_sma_status(days_past_due)
        asset_classification = get_asset_classification(days_past_due)
        provisioning_percentage = npa_mapping[asset_classification]
        provisioning_amount = round(overdue_amount * provisioning_percentage, 2)
        recovery_amount = round(random.uniform(0, overdue_amount), 2)
        recovery_percentage = round((recovery_amount / loan_amount) * 100, 2)
        write_off_amount = round(overdue_amount - recovery_amount, 2) if asset_classification in ['Loss', 'Doubtful-3'] else 0
        restructuring_status = random.choice(['Yes', 'No'])
        restructuring_date = fake.date_between(start_date='-3y', end_date='today') if restructuring_status == 'Yes' else None

        data.append([
            loan_id, loan_type, loan_amount, overdue_amount, interest_rate, tenure_months, disb_date, borrower_id,
            credit_score, annual_income, industry, region, days_past_due, sma_status, asset_classification,
            provisioning_percentage, provisioning_amount, recovery_amount, recovery_percentage, write_off_amount,
            restructuring_status, restructuring_date, branch_name, loan_purpose
        ])
        
    columns = [
        'Loan_ID', 'Loan_Type', 'Loan_Amount', 'Overdue_Amount', 'Interest_Rate', 'Tenure_Months', 'Disbursement_Date',
        'Borrower_ID', 'Credit_Score', 'Annual_Income', 'Industry', 'Region', 'Days_Past_Due', 'SMA_Status',
        'Asset_Classification', 'Provisioning_Percentage', 'Provisioning_Amount', 'Recovery_Amount', 'Recovery_Percentage',
        'Write_Off_Amount', 'Restructuring_Status', 'Restructuring_Date', 'Branch_Name', 'Loan_Purpose'
    ]
    
    return pd.DataFrame(data, columns=columns)

In [15]:
# Generate
df_npa = generate_npa_data()

# Save
df_npa.to_csv('NPA_Data.csv', index=False)

In [17]:
print("10000 rows generated and saved to 'NPA_Data.csv'")

10000 rows generated and saved to 'NPA_Data.csv'
